In [10]:
# Instalación de dependencias
#!pip install cohere pandas tqdm

In [1]:
import cohere
import json
import os
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import random

# Inicializar el cliente de Cohere
co = cohere.Client('YOUR_API_KEY_HERE')  # Reemplazar con tu API key
MODEL_NAME = 'command-r-plus' 

In [ ]:
# Función para cargar datos
def load_json_data(data_dir: str):
    """Cargar todos los archivos JSON del directorio de datos."""
    all_data = []
    for file in os.listdir(data_dir):
        if file.endswith('.json'):
            with open(os.path.join(data_dir, file), 'r', encoding='utf-8') as f:
                data = json.load(f)
                all_data.extend(data)
    return all_data

# Cargar los datos - Ajustado a tu ruta específica
data_dir = '../../data/raw'  # Esta es la carpeta donde están tus JSON
stories = load_json_data(data_dir)
print(f"Se cargaron {len(stories)} historias")

# Mostrar un ejemplo de la primera historia
if stories:
    print("\nEjemplo de la primera historia:")
    print(json.dumps(stories[0], indent=2, ensure_ascii=False))

Se cargaron 573 historias

Ejemplo de la primera historia:
{
  "id": "ar_01",
  "nombre": "La Difunta Correa",
  "pais": "Argentina",
  "descripcion": "La Difunta Correa es una figura semilegendaria venerada en Argentina, especialmente en la región de Cuyo. Se cree que fue una joven llamada Deolinda Correa que murió de sed en el desierto sanjuanino durante las guerras civiles del siglo XIX, mientras buscaba a su esposo. Su historia ha dado origen a un culto popular, con santuarios y ofrendas en su honor.",
  "historia": "Durante las guerras civiles argentinas del siglo XIX, Deolinda Correa emprendió un viaje por el desierto de San Juan con su hijo en brazos, en busca de su esposo reclutado a la fuerza. Agotada y sin agua, Deolinda murió en el camino. Días después, unos arrieros encontraron su cuerpo sin vida, pero su hijo seguía vivo, amamantándose de su madre. Este milagro dio origen a la veneración de la Difunta Correa, y su tumba en Vallecito, San Juan, se convirtió en un santuario 

# Getting the most common themes in literature
Now let's access this site: https://prowritingaid.com/themes-in-literature

In [3]:
story_themes = [
    "Abuso de poder", "Adulterio", "Adversidad", "Envejecimiento", "Alienación", "Ambiciones",
    "Sueño americano", "Arrogancia", "Arte", "Autonomía", "Belleza", "Creencias", "Traición",
    "Valentía", "Capitalismo", "Celebración", "Azar", "Cambio versus tradición",
    "Caos y orden", "Carácter", "Infancia", "Ciclo de la vida", "Clase social", "Cambio climático",
    "Colonialismo", "Mayoría de edad", "Sentido común", "Comunicación", "Compañerismo",
    "Conservación", "Conspiración", "Convención y rebelión", "Corrupción", "Coraje",
    "Creación", "Crimen", "Oscuridad y luz", "Muerte", "Dedicación", "Democracia",
    "Depresión", "Deseo", "Desesperación", "Destino", "Decepción", "Desilusión",
    "Desplazamiento", "Sueños", "Economía", "Educación", "Empoderamiento", "Amor eterno",
    "Fracaso", "Fe", "Fama", "Familia", "Destino", "Miedo", "Feminismo", "Amor prohibido",
    "Perdón", "Libre albedrío", "Libertad", "Amistad", "Plenitud", "Futuro",
    "Derechos LGBT", "Género", "Dios", "Bien contra mal",
    "Gobierno", "Gratitud", "Codicia", "Crecimiento", "Culpa", "Felicidad", "Trabajo duro",
    "Odio", "Salud", "Corazón roto", "Héroe", "Heroísmo", "Historia", "Honestidad", "Honor",
    "Esperanza", "Humanidad", "Naturaleza humana", "Humildad", "Humor", "Hipocresía", "Identidad",
    "Ideología", "Imaginación", "Inmortalidad", "Imperialismo", "Imposibilidad", "Individualidad",
    "Desigualdad", "Injusticia", "Inocencia", "Inspiración", "Aislamiento", "Celos", "Alegría",
    "Justicia", "Amabilidad", "Conocimiento", "Ley", "Legado", "Vida", "Soledad", "Pérdida",
    "Amor", "Lealtad", "Locura", "Manipulación", "Materialismo", "Madurez", "Medicina",
    "Recuerdos", "Misericordia", "Dinero", "Moralidad", "Maternidad", "Música", "Nacionalismo", "Naturaleza",
    "Necesidad", "Negligencia", "Año nuevo", "Normalidad", "No rendirse", "Unidad", "Oportunidad",
    "Opresión", "Optimismo", "Superación", "Pasión", "Paz", "Presión social", "Perfección",
    "Perseverancia", "Desarrollo personal", "Política", "Pobreza", "Poder", "Oración", "Prejuicio",
    "Orgullo", "Progreso", "Propaganda", "Propósito", "Realismo", "Realidad", "Rebelión",
    "Renacimiento", "Redención", "Arrepentimiento", "Relaciones", "Religión", "Represión", "Resistencia",
    "Venganza", "Revolución", "Sacrificio", "Tristeza", "Sátira", "Ciencia", "Autoconciencia",
    "Autodisciplina", "Autosuficiencia", "Autopreservación", "Simplicidad", "Pecado", "Sociedad",
    "Soledad", "Estoicismo", "Subjetividad", "Sufrimiento", "Vigilancia", "Supervivencia",
    "Compasión", "Tecnología", "Tentación", "Tiempo", "Tolerancia", "Totalitarismo", "Tragedia",
    "Viaje", "Confianza", "Verdad", "Amor incondicional", "Universo", "Amor no correspondido", "Desinterés",
    "Valor", "Vanidad", "Vicios", "Violencia", "Virtud", "Guerra", "Desperdicio", "Riqueza", "Fuerza de voluntad",
    "Ganar y perder", "Sabiduría", "Trabajo", "Luchas de la clase trabajadora", "Xenofobia", "Juventud"
]


In [11]:
def create_topic_prompt(theme, original_story):
    """Crea un prompt para generar tramas basadas en un tema y una historia original."""
    prompt = f"""Eres un chatbot experto en crear tramas CORTAS de mitos y leyendas de España, Portugal y América Latina.
    La respuesta debe ser directa, sin explicaciones, sin títulos y de frente la numeracion.
    Prohibido superar 30 palabras por cada trama.
    Por ejemplo: 
    <User>: Porfavor dame una trama de una historia basado en el siguiente contexto:
    Tema: {theme}
    Protagonista: {original_story['nombre']}
    País: {original_story['pais']}
    Descripción: {original_story['descripcion']}
    Historia Original: {original_story['historia']}

    <Assistant>:
    1. Un curandero que descubre el poder de las hierbas sagradas en la selva amazónica.
    2. Un chamán que puede comunicarse con los espíritus de los ancestros.

    Ahora es tu turno. Basándote en el contexto proporcionado, crea solo 2 tramas diferentes para el protagonista, dame la numeración directamente.
    Responde de manera creativa e innovadora y no repitas los ejemplos. ¡Cuanto más única y diferente sea tu respuesta, mejor ¡No pongas títulos a tus historias!"""
    return prompt


def generate_tramas(co, theme, original_story):
    """Genera una trama basada en un tema y una historia original."""
    prompt = create_topic_prompt(theme, original_story)
    response = co.generate(
        model=MODEL_NAME,  # Especificamos el modelo aquí
        prompt=prompt,
        max_tokens=128,
        temperature=0.7,
        k=0,
        p=0.8,
        frequency_penalty=0.1,
        presence_penalty=0.1
    )
    return response.generations[0].text.strip()

# Función para generar y guardar tramas
def generate_and_save_tramas(co, stories, themes):
    """Genera tramas para cada historia usando 3 temas aleatorios diferentes."""
    all_tramas = []
    
    for story in tqdm(stories, desc="Generando tramas"):
        # Seleccionar 3 temas aleatorios diferentes
        selected_themes = random.sample(themes, 3)
        
        # Para cada tema seleccionado, generar una trama
        for theme in selected_themes:
            # Generar trama
            trama = generate_tramas(co, theme, story)
            
            # Guardar
            all_tramas.append({
                'id': story['id'],
                'nombre': story['nombre'],
                'pais': story['pais'],
                'Theme': theme,
                'Trama': trama
            })
    
    # Crear DataFrame y guardar
    df_tramas = pd.DataFrame(all_tramas)
    df_tramas.to_csv('../../data/generated/tramas.csv', index=False)
    
    return df_tramas

In [12]:
# Probar la generación de tramas
df_tramas = generate_and_save_tramas(co, stories, story_themes)
print("\nEjemplo de tramas generadas:")
print(df_tramas.head())

Generando tramas:  17%|█▋        | 98/573 [34:39<2:48:00, 21.22s/it]


KeyboardInterrupt: 

In [ ]:
def create_story_prompt(theme, trama, original_story):
    """Genera un prompt que será usado más adelante para crear una historia basada en una leyenda iberoamericana."""

    prompt = f"""Eres un generador de prompts para crear historias inspiradas en mitos y leyendas iberoamericanas.

    Tema: {theme}
    Trama: {trama}
    Leyenda base: {original_story['nombre']} ({original_story['pais']})
    Descripción: {original_story['descripcion']}
    Historia original: {original_story['historia']}

    Ejemplos de prompt generado:
    - "Crea una historia basada en la leyenda de El Cadejo, con la trama 'un joven enfrenta sus miedos durante un ritual nocturno' y el tema 'valentía frente a lo desconocido'. ¿Cómo cambia su vida luego de esa noche?"
    - "A partir de la leyenda de La Llorona, crea una historia donde una niña escucha voces en el río y descubre un antiguo secreto familiar. El tema es 'culpa y redención'."
    - "Escribe una historia inspirada en la leyenda del Nahual, siguiendo la trama 'un niño descubre su conexión con los animales del bosque' y el tema 'identidad y transformación'."
    - "Crea una historia basada en la leyenda de El Sombrerón, con la trama 'una adolescente desafía las supersticiones de su pueblo' y el tema 'conocimiento frente al miedo'. ¿Qué consecuencias trae su curiosidad?"

    Ahora genera un prompt similar, siguiendo este estilo:"""

    return prompt


In [ ]:
# Función para generar tramas
def generate_tramas(co, theme, original_story):
    """Genera 5 tramas basadas en un tema y una historia original."""
    prompt = create_topic_prompt(theme, original_story)
    response = co.generate(
        prompt=prompt,
        max_tokens=512,
        temperature=0.7,
        k=0,
        p=0.9,
        frequency_penalty=0.1,
        presence_penalty=0.1,
        stop_sequences=[],
        return_likelihoods='NONE'
    )
    return response.generations[0].text.strip()

# Función para generar instrucción-pregunta
def generate_story_prompt(co, theme, trama, original_story):
    """Genera una instrucción-pregunta basada en el tema, trama y historia original."""
    prompt = create_story_prompt(theme, trama, original_story)
    response = co.generate(
        prompt=prompt,
        max_tokens=512,
        temperature=0.7,
        k=0,
        p=0.9,
        frequency_penalty=0.1,
        presence_penalty=0.1,
        stop_sequences=[],
        return_likelihoods='NONE'
    )
    return response.generations[0].text.strip()

# Función para generar la historia final
def generate_story(co, instruccion, original_story):
    """Genera una historia basada en la instrucción-pregunta y la historia original."""
    final_prompt = f"""Eres un experto narrador de mitos y leyendas iberoamericanas.

Contexto Original:
Título: {original_story['nombre']}
País: {original_story['pais']}
Descripción: {original_story['descripcion']}
Historia Original: {original_story['historia']}

Instrucción: {instruccion}

Por favor, escribe una historia corta (máximo dos párrafos) que:
1. Siga la instrucción proporcionada
2. Mantenga la esencia cultural y mágica de la historia original
3. Sea apropiada para niños
4. Tenga un inicio, desarrollo y final claros
5. Preserve elementos clave de la historia original

Historia:"""
    
    response = co.generate(
        prompt=final_prompt,
        max_tokens=512,
        temperature=0.7,
        k=0,
        p=0.9,
        frequency_penalty=0.1,
        presence_penalty=0.1,
        stop_sequences=[],
        return_likelihoods='NONE'
    )
    return response.generations[0].text.strip()